<img src="logo.png" alt="Vegeta" width="240">

# Robot dog — a quadruped for patrols, stairs and building inspection

A 13 kg quadruped — **Cerberus**, the dog of the underworld — for patrolling buildings that may not be
safe for a person: stairs, slopes, a drop from a step, a 5 kg sensor pack on the flat deck on its back.
It is also the first four-legged unit of the Sikarian line: every segment of the **Myropods** (notebooks
17–19) is a four-legged walker that stands on its own, so the legs, the actuator choices and the gait
model are worked out here first, on one body. Actuators come from the shared library
`designs/actuators.py`. This notebook is the dog's mechanics end to end:

```
CAD (Dedalus): body shell with the deck, head, four two-link legs ─► masses, standing geometry
gaits (duty factor) + payload ─► leg forces ─► joint torques over the stance (planar two-link leg) ─► motor choice
stairs, slopes, a drop from a step ─► operating cases ─► static FEA of the upper and lower leg (Talos), pins by hand
lower-leg modes ─► frequency diagram against stride harmonics
missions (patrol, carrying the pack, stairs) ─► foot-force histories ─► rainflow (Chronos) ─► fatigue of the lower leg ─► life
the trot and a walk over open terrain as sequences of leg angles ─► movies ─► print the upper leg (Mellonia) ─► JSON record
```

Every input is explicit and coarse (motor data, friction, an assumed S-N curve). Compare, record, then
walk the real dog up a real staircase.

In [ ]:
import json, math, shutil
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from vegeta import dedalus, talos, chronos, mellonia
from vegeta.dedalus import viz as dviz
from vegeta.talos import viz as tviz
from vegeta.mellonia import viz as mviz
from vegeta.mellonia.examples import GENERIC_PLA_0_2MM
import sys; sys.path.insert(0, "designs"); import actuators as act          # the shared actuator / motor / joint library (designs/actuators.py)

RUNS = Path("_runs/robot_dog"); shutil.rmtree(RUNS, ignore_errors=True); RUNS.mkdir(parents=True)
design_file = RUNS / "robot_dog.py"
shutil.copy(Path("designs/robot_dog.py"), design_file)
dog_design = dedalus.load_design(f"{design_file}:RobotDog")
pd.DataFrame(dog_design.params.table()).set_index("name")

## 1. The dog

In [ ]:
p = dog_design.resolve()
dog = dog_design.generate()
body = dog_design.generate(part="body")
upper = dog_design.generate(part="upper_leg")
lower = dog_design.generate(part="lower_leg")
cad = {k: g.export(RUNS / "cad" / k, stl_tolerance=0.05) for k, g in (("dog", dog), ("body", body), ("upper_leg", upper), ("lower_leg", lower))}
H_STAND = dog_design.standing_height(p)          # hip height above the ground [mm]
print(f"standing height {H_STAND:.0f} mm | foot {dog_design.foot_x(p):.0f} mm ahead of the hip | leg fully stretched {p['upper_leg_length'] + p['lower_leg_length']:.0f} mm")
dviz.show(dviz.plot3d(dog))

In [ ]:
y_leg = p["body_width"] / 2 + p["hip_boss_length"] + p["upper_leg_thickness"] / 2 + 1.0
fig = dviz.plot_sections(dog, normal="y", positions=[0.0, y_leg], cols=2)          # body centre plane, leg plane
fig = dviz.plot_sections(dog, normal="x", positions=[p["hip_x"], 0.0, -p["hip_x"]], cols=3)

In [ ]:
RHO_PA12CF = 1.1e-3                      # g/mm^3, printed PA12-CF
SHELL_T = 3.0                            # mm, the body is a printed shell of this thickness over a frame (the CAD is solid)
m_shell = body.surface_area * SHELL_T * RHO_PA12CF
parts = pd.DataFrame([
    ("body shell, 3 mm PA12-CF (from CAD area)", m_shell),
    ("aluminium frame + deck plate", 1400.0),
    ("upper legs 4x (from CAD)", 4 * upper.volume * RHO_PA12CF),
    ("lower legs 4x (from CAD, CF-nylon)", 4 * lower.volume * 1.15e-3),
    ("actuators 12x (hip roll, hip pitch, knee; 480 g each)", 12 * 480.0),
    ("battery 10S 10 Ah", 2200.0),
    ("computer, radios, IMU, cameras", 650.0),
    ("wiring, bearings, pins, feet", 500.0),
], columns=["part", "mass_g"]).set_index("part")
M_DOG = parts["mass_g"].sum() / 1000
PAYLOAD_ALLOW_KG = 5.0                    # payload allowance on the deck (input: a sensor pack, a radio relay, tools)
G = 9.81
W_DOG, W_LOADED = M_DOG * G, (M_DOG + PAYLOAD_ALLOW_KG) * G
print(f"dog {M_DOG:.2f} kg | with the full payload allowance {M_DOG + PAYLOAD_ALLOW_KG:.2f} kg | static foot load {W_DOG / 4:.0f} N ({W_LOADED / 4:.0f} N loaded)")
parts.round(0)

## 2. Standing, walking, trotting: foot forces and joint torques

A stance leg carries a half-sine ground reaction over its contact time. With duty factor β (the fraction
of the stride a foot is on the ground) the mean number of feet down is 4β, so the peak vertical force per
foot is `F_peak = (π/2) · W / (4β)`: a walk (β = 0.75) ~0.5 W, a trot (β = 0.5) ~0.8 W, a bound (β = 0.3)
~1.3 W. Standing is static: W/4 each, shifted fore/aft by the centre of gravity. Braking and climbing add a
longitudinal component (`Fx ≈ ±0.3 Fz` at a trot, more on stairs).

The joint torques follow from the moment of the foot force about each joint — a planar two-link leg, swept
over a stance (the foot moves from ahead of the hip to behind it) at the standing height.

In [ ]:
GAITS = {"walk": 0.75, "trot": 0.50, "bound": 0.30}
def foot_peak(W, beta): return (math.pi / 2) * W / (4 * beta)
X_CG = 25.0                               # dog's CG ahead of the body centre [mm] (head, computer up front; input)
WHEELBASE = 2 * p["hip_x"]
def static_split(W, x_cg):                # front/rear share of the weight from the CG position
    front = 0.5 + x_cg / WHEELBASE
    return {"front_N": W * front / 2, "rear_N": W * (1 - front) / 2}
gait_tab = pd.DataFrame({g: {"duty": b, "feet down (mean)": 4 * b, "peak foot force, dog only [N]": foot_peak(W_DOG, b),
                             "peak foot force, with payload [N]": foot_peak(W_LOADED, b), "F/W per foot": (math.pi / 2) / (4 * b)} for g, b in GAITS.items()}).T
gait_tab.round(2)

In [ ]:
L1, L2 = p["upper_leg_length"], p["lower_leg_length"]
def leg_ik(x_foot, z_foot):
    """Hip and knee angles (upper leg behind vertical, lower leg ahead of vertical, knee back) for a foot at (x, z) relative to the hip."""
    r = math.hypot(x_foot, z_foot)
    if r > L1 + L2 - 1e-6 or r < abs(L1 - L2) + 1e-6:
        return None
    c_knee = (L1**2 + L2**2 - r**2) / (2 * L1 * L2)          # interior knee angle
    knee_int = math.acos(max(-1, min(1, c_knee)))
    # angle of the hip->foot line from straight down, and the angle between the upper leg and that line
    phi = math.atan2(x_foot, -z_foot)
    alpha = math.acos(max(-1, min(1, (L1**2 + r**2 - L2**2) / (2 * L1 * r))))
    a1 = -(phi - alpha)                                       # upper leg behind vertical is positive
    a2 = math.pi - knee_int - a1                              # lower leg ahead of vertical
    return math.degrees(a1), math.degrees(a2)

def joint_torques(x_foot, z_foot, fx, fz):
    """Hip-pitch and knee torques [N m] holding a foot force (fx forward, fz up on the foot) at the given foot position [mm]."""
    ang = leg_ik(x_foot, z_foot)
    if ang is None:
        return float("nan"), float("nan")
    a1, a2 = map(math.radians, ang)
    knee = np.array([-L1 * math.sin(a1), -L1 * math.cos(a1)])
    foot = np.array([x_foot, z_foot])
    F = np.array([fx, fz])
    tau_hip = (foot[0] * F[1] - foot[1] * F[0]) / 1000
    tau_knee = ((foot[0] - knee[0]) * F[1] - (foot[1] - knee[1]) * F[0]) / 1000
    return tau_hip, tau_knee

STRIDE = 0.45 * H_STAND                       # foot travel during one stance at a trot [mm] (input: ~ half the leg height)
xs = np.linspace(STRIDE / 2, -STRIDE / 2, 41)   # the foot lands ahead and leaves behind
QDD = act.get("qdd 24 Nm")                      # from the shared library: a quasi-direct-drive leg module
MOTOR = {"name": QDD.key, "continuous_Nm": QDD.rated_Nm, "peak_Nm": QDD.stall_Nm}
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
torque_rows = {}
for (gname, beta), ls in zip(GAITS.items(), ("-", "--", ":")):
    for W, tag, col in ((W_DOG, "dog", "#1565c0"), (W_LOADED, "with payload", "#c62828")):
        Fz = foot_peak(W, beta) * np.sin(np.pi * np.linspace(0, 1, len(xs)))      # half-sine over the stance
        Fx = 0.3 * Fz * np.cos(np.pi * np.linspace(0, 1, len(xs)))               # braking then pushing
        th = np.array([joint_torques(x, -H_STAND, fx, fz) for x, fx, fz in zip(xs, Fx, Fz)])
        axes[0].plot(xs, th[:, 0], ls, color=col, label=f"{gname}, {tag}"); axes[1].plot(xs, th[:, 1], ls, color=col, label=f"{gname}, {tag}")
        torque_rows[f"{gname}, {tag}"] = {"peak hip [Nm]": float(np.nanmax(np.abs(th[:, 0]))), "peak knee [Nm]": float(np.nanmax(np.abs(th[:, 1])))}
for ax, name in zip(axes, ("hip pitch", "knee")):
    ax.axhline(MOTOR["continuous_Nm"], color="k", lw=0.8, ls="-."); ax.axhline(-MOTOR["continuous_Nm"], color="k", lw=0.8, ls="-.")
    ax.axhline(MOTOR["peak_Nm"], color="k", lw=0.8); ax.axhline(-MOTOR["peak_Nm"], color="k", lw=0.8)
    ax.set(title=f"{name} torque over a stance (foot ahead → behind)", xlabel="foot position ahead of the hip [mm]", ylabel="N m"); ax.grid(alpha=0.3)
axes[0].legend(fontsize=7)
torques = pd.DataFrame(torque_rows).T
torques["hip / peak motor"] = torques["peak hip [Nm]"] / MOTOR["peak_Nm"]; torques["knee / peak motor"] = torques["peak knee [Nm]"] / MOTOR["peak_Nm"]
torques.round(2)

## 3. The mission: stairs and slopes with the payload on the back

A standard flight of stairs (170 mm risers, 280 mm treads, a 31° slope). Three checks, all geometric or
static: the leg must lift a foot over the riser with a margin (inverse kinematics against the joint ranges),
the feet must hold on the slope (friction, with the weight shifted to the rear pair), and the loaded dog
must not tip backwards on the slope (the CG projection must stay inside the support polygon, with the
payload's CG above the deck).

In [ ]:
RISER, TREAD = 170.0, 280.0                        # stairs [mm] (input)
SLOPE_DEG = math.degrees(math.atan(RISER / TREAD))
CLEARANCE = 40.0                                   # foot lift above the riser edge [mm] (input)
MU_FEET = {"dry concrete": 0.7, "wet tiles": 0.35, "soot on stone": 0.3}
HIP_RANGE, KNEE_RANGE = (-30.0, 110.0), (0.0, 150.0)   # joint ranges of the real actuators [deg] (input)
PAYLOAD_CG_ABOVE_DECK = 60.0                       # payload CG above the deck surface [mm] (input)
DECK_Z = H_STAND + 8.0 + p["body_height"] / 2 - p["deck_depth"]
H_CG_DOG = H_STAND + 8.0                           # body centre
H_CG_LOADED = (M_DOG * H_CG_DOG + PAYLOAD_ALLOW_KG * (DECK_Z + PAYLOAD_CG_ABOVE_DECK)) / (M_DOG + PAYLOAD_ALLOW_KG)

# (a) foot lift: the swing foot must reach z = -(H_STAND - RISER - CLEARANCE) at x = +TREAD/2 while the body stays level
lift_ok = leg_ik(TREAD / 2, -(H_STAND - RISER - CLEARANCE))
# (b) a crouch: the lowest standing height within the joint ranges (to get under the hearth lintel) and the tallest reach
crouch = min((z for z in np.arange(100, H_STAND, 5) if (a := leg_ik(0.0, -z)) and HIP_RANGE[0] <= a[0] <= HIP_RANGE[1] and KNEE_RANGE[0] <= a[1] <= KNEE_RANGE[1]), default=None)
# (c) slope: weight split and the friction needed, plus the backward-tipping margin
def slope_check(W, h_cg, theta_deg, mu):
    th = math.radians(theta_deg)
    shift = h_cg / WHEELBASE * math.tan(th)                                  # share moved to the downhill (rear) pair
    rear = W * math.cos(th) * (0.5 + shift); front = W * math.cos(th) * (0.5 - shift)
    needed = W * math.sin(th)                                                  # total tangential force to hold / climb
    available = mu * (rear + front)
    tip_margin = WHEELBASE / 2 - h_cg * math.tan(th)                           # CG projection inside the rear feet [mm]
    return {"rear pair [N]": rear, "front pair [N]": front, "tangential needed [N]": needed, "friction available [N]": available,
            "friction SF": available / needed, "tipping margin [mm]": tip_margin}
rows = {}
for surf, mu in MU_FEET.items():
    rows[f"dog only, {surf}"] = slope_check(W_DOG, H_CG_DOG, SLOPE_DEG, mu)
    rows[f"with payload, {surf}"] = slope_check(W_LOADED, H_CG_LOADED, SLOPE_DEG, mu)
stairs = pd.DataFrame(rows).T
print(f"stairs slope {SLOPE_DEG:.1f}° | foot over the riser needs hip {lift_ok[0]:.0f}°, knee {lift_ok[1]:.0f}° (ranges {HIP_RANGE}, {KNEE_RANGE}) | lowest crouch {crouch} mm hip height")
print(f"CG height: dog {H_CG_DOG:.0f} mm, loaded {H_CG_LOADED:.0f} mm | deck at {DECK_Z:.0f} mm")
stairs.round(2)

## 4. A drop from a step: the hardest landing

Stepping off a kerb or a hearth (`DROP_M`) the dog lands on its front pair. The legs absorb the fall over a
compression stroke `S_LEG` (the knees fold under impedance control); the mean leg force is the kinetic energy
over the stroke, plus the weight, and a half-sine pulse peaks at π/2 of the mean. Payload included.

In [ ]:
DROP_M, S_LEG = 0.30, 0.080               # drop height and leg compression stroke [m] (inputs)
def landing_force(mass, n_legs=2):
    mean = mass * G * (DROP_M / S_LEG + 1) / n_legs
    return {"mean per leg [N]": mean, "peak per leg [N]": mean * math.pi / 2, "contact time [ms]": 2 * S_LEG / math.sqrt(2 * G * DROP_M) * 1000}
landing = pd.DataFrame({"dog": landing_force(M_DOG), "with payload": landing_force(M_DOG + PAYLOAD_ALLOW_KG)}).T
landing.round(1)

## 5. Static strength of the legs under the operating cases (Talos)

The lower leg is analysed in its own frame (knee bore fixed, force on the foot), the upper leg in its (hip
bore fixed, force on the knee bore — the pin force the lower leg hands over). Ground forces (`fx` forward,
`fz` up) are rotated into each leg's frame with the standing angles. Six cases per leg: standing, the trot
and bound peaks, the stairs push (large `fx`), the loaded trot, and the drop landing. Pins by hand (bending
over the clevis gap, double shear).

In [ ]:
PA12CF = talos.Material("PA12-CF (printed)", youngs_modulus=3500.0, poissons_ratio=0.40, density=1.1e-9, yield_strength=60.0,
                        source="nominal datasheet, flat orientation; assume 30 % less across layers")
CFNYLON = talos.Material("CF-nylon rod (printed, continuous fibre)", youngs_modulus=9000.0, poissons_ratio=0.38, density=1.15e-9, yield_strength=120.0,
                         source="assumed from supplier data; coupon tests needed")
STEEL_PIN = talos.Material("steel pin (C45)", youngs_modulus=210000.0, poissons_ratio=0.30, density=7.85e-9, yield_strength=400.0, source="handbook")
a1, a2 = math.radians(p["hip_angle_deg"]), math.radians(p["knee_angle_deg"])
def to_lower(fx, fz):    # ground force -> lower-leg frame (the lower leg was rotated by -a2 about y)
    return fx * math.cos(a2) - fz * math.sin(a2), fx * math.sin(a2) + fz * math.cos(a2)
def to_upper(fx, fz):    # ground force -> upper-leg frame (rotated by +a1 about y)
    return fx * math.cos(a1) + fz * math.sin(a1), -fx * math.sin(a1) + fz * math.cos(a1)

t, w, L1k = p["upper_leg_thickness"], p["upper_leg_width"], p["upper_leg_length"]
rp = p["pin_diameter"] / 2 + 0.3
LOWER_REGIONS = [talos.SurfacesInBox("knee", (-rp, -t, -rp, rp, t, rp)),
                 talos.SurfacesInBox("foot", (-p["foot_diameter"], -p["foot_diameter"], -L2 - p["foot_diameter"], p["foot_diameter"], p["foot_diameter"], -L2 + p["foot_diameter"]))]
UPPER_REGIONS = [talos.SurfacesInBox("hip", (-rp, -t, -rp, rp, t, rp)), talos.SurfacesInBox("knee", (-rp, -t, -L1k - rp, rp, t, -L1k + rp))]
def lower_model(loads, name, masses=()):
    return talos.StructuralModel(cad["lower_leg"].artifacts["step"], "mm-N-MPa", CFNYLON, LOWER_REGIONS, [talos.FixedSupport("knee")], loads,
                                 talos.MeshSettings(element_size=3.0, min_element_size=1.0, curvature_points=12), name=name, masses=list(masses))
def upper_model(loads, name):
    return talos.StructuralModel(cad["upper_leg"].artifacts["step"], "mm-N-MPa", PA12CF, UPPER_REGIONS, [talos.FixedSupport("hip")], loads,
                                 talos.MeshSettings(element_size=3.5, min_element_size=1.2), name=name)

trot, bound = foot_peak(W_DOG, GAITS["trot"]), foot_peak(W_DOG, GAITS["bound"])
loaded_trot = foot_peak(W_LOADED, GAITS["trot"])
stair_rear = stairs.loc["with payload, dry concrete", "rear pair [N]"] / 2
CASES = {   # ground forces on one foot: fx forward (+ = the foot pushes the dog forward), fz up
    "standing, loaded": dict(fx=0.0, fz=static_split(W_LOADED, X_CG)["front_N"]),
    "trot peak": dict(fx=0.3 * trot, fz=trot),
    "bound peak": dict(fx=0.4 * bound, fz=bound),
    "stairs push, loaded (rear foot)": dict(fx=stair_rear * math.tan(math.radians(SLOPE_DEG)) * 1.5, fz=stair_rear * 1.2),
    "trot peak, loaded": dict(fx=0.3 * loaded_trot, fz=loaded_trot),
    "drop landing, loaded": dict(fx=0.2 * landing.loc["with payload", "peak per leg [N]"], fz=landing.loc["with payload", "peak per leg [N]"]),
}
lower_model([talos.Force("foot", fz=1.0)], "mesh").mesh(RUNS / "lower_mesh", progress=True)
upper_model([talos.Force("knee", fz=1.0)], "mesh").mesh(RUNS / "upper_mesh", progress=True)
lower_results, upper_results = {}, {}
for name, f in tqdm(CASES.items()):
    lx, lz = to_lower(f["fx"], f["fz"]); ux, uz = to_upper(f["fx"], f["fz"])
    key = name.replace(" ", "_").replace(",", "").replace("(", "").replace(")", "")
    lm = lower_model([talos.Force("foot", fx=lx, fz=lz)], key); lm.mesh(RUNS / f"lower_{key}"); lower_results[name] = lm.solve(RUNS / f"lower_{key}")
    um = upper_model([talos.Force("knee", fx=ux, fz=uz)], key); um.mesh(RUNS / f"upper_{key}"); upper_results[name] = um.solve(RUNS / f"upper_{key}")
print({k: r.status for k, r in lower_results.items()})

In [ ]:
def pin_bending(F, lever_mm, d_mm): return F * lever_mm / (math.pi * d_mm**3 / 32)
def pin_shear(F, d_mm): return F / (2 * math.pi * d_mm**2 / 4)        # double shear
lever = (t / 2 + 3.0) / 2                                             # half the clevis gap: plate thickness to the boss
pins = {}
for name, f in CASES.items():
    F = math.hypot(f["fx"], f["fz"])
    sig = math.sqrt(pin_bending(F, lever, p["pin_diameter"])**2 + 3 * pin_shear(F, p["pin_diameter"])**2)
    pins[name] = {"load": f"{F:.0f} N resultant", "max_von_mises_MPa": sig, "SF_yield": STEEL_PIN.yield_strength / sig, "deflection_mm": float("nan")}
def rows(results, label):
    return pd.DataFrame({f"{label} — {k}": {"load": f"fx {CASES[k]['fx']:.0f} N, fz {CASES[k]['fz']:.0f} N", "max_von_mises_MPa": r.metrics.get("max_von_mises"),
                                            "SF_yield": r.metrics.get("safety_factor_yield"), "deflection_mm": r.metrics.get("max_displacement")} for k, r in results.items()}).T
static = pd.concat([rows(lower_results, "lower leg"), rows(upper_results, "upper leg"), pd.DataFrame(pins).T.rename(index=lambda k: f"pins — {k}")])
fig, ax = plt.subplots(figsize=(9, 6))
static["SF_yield"].astype(float).plot.barh(ax=ax, color=["#1565c0"] * len(CASES) + ["#2e7d32"] * len(CASES) + ["#6d4c41"] * len(CASES))
ax.axvline(1.5, color="#c62828", ls="--", label="SF 1.5 (printed parts, flat orientation)"); ax.set(xlabel="safety factor on yield", xscale="log"); ax.legend(); ax.grid(alpha=0.3, axis="x")
worst = static.loc[[k for k in static.index if k.startswith("lower")], "SF_yield"].astype(float).idxmin()
static.round(2)

In [ ]:
tviz.show(tviz.plot_results(lower_results[worst.split(" — ")[1]], field="von_mises"))

## 6. Lower-leg modes against the stride harmonics

The lower leg (knee held, the foot free) rings when a foot strikes. Strides excite it at the stride frequency
and its harmonics; the foot-strike impulse itself is broad. The diagram is the dog's Campbell diagram: the
excitation lines scale with speed (stride frequency from a Froude-scaled stride length) instead of rpm.

In [ ]:
FOOT_MASS_T = 0.045e-3                                                        # rubber foot + sensor at the end [t]
modal = lower_model([], "lower_modal", masses=[talos.PointMass("foot", FOOT_MASS_T)])
modal.mesh(RUNS / "lower_modal"); modes = modal.solve_modes(RUNS / "lower_modal", n_modes=4)
freqs = modes.metrics["frequencies_hz"]
K_VIRTUAL = 4000.0                                                             # virtual leg stiffness under impedance control [N/m] (input)
f_bounce = math.sqrt(2 * K_VIRTUAL / M_DOG) / (2 * math.pi)                    # body bounce at a trot: two legs down
speeds = np.linspace(0.3, 3.0, 28)
stride_len = 0.9 * H_STAND / 1000 * np.sqrt(speeds / math.sqrt(G * H_STAND / 1000)) * 2 + 0.2   # Froude scaling of the stride length [m] (coarse)
f_stride = speeds / stride_len
fig, ax = plt.subplots(figsize=(8, 4.5))
for k in (1, 2, 4, 8):
    ax.plot(speeds, k * f_stride, label=f"stride x{k}")
for i, f in enumerate(freqs):
    ax.axhline(f, color="#c62828", ls="--", lw=0.8); ax.text(speeds[0], f, f" lower-leg mode {i + 1}: {f:.0f} Hz", color="#c62828", va="bottom", fontsize=8)
ax.axhline(f_bounce, color="#1565c0", ls="-.", lw=0.8); ax.text(speeds[-1], f_bounce, f"body bounce {f_bounce:.1f} Hz ", color="#1565c0", ha="right", va="bottom", fontsize=8)
for name, v in (("walk", 0.6), ("trot", 1.5), ("bound", 2.6)):
    ax.axvline(v, color="#888", ls=":"); ax.text(v, 0.4, f" {name}", rotation=90, va="bottom", fontsize=8)
ax.set(xlabel="speed [m/s]", ylabel="frequency [Hz]", yscale="log", ylim=(0.3, 3000), title="frequency diagram: stride harmonics vs the lower-leg modes")
ax.legend(fontsize=7, loc="upper left"); ax.grid(alpha=0.3, which="both")
structure = chronos.Structure(tuple(freqs), damping_ratio=0.03)
print(f"lower-leg modes {[round(f, 1) for f in freqs]} Hz | body bounce {f_bounce:.1f} Hz | stride at a trot {np.interp(1.5, speeds, f_stride):.1f} Hz, margin to the nearest mode {structure.margin(8 * np.interp(1.5, speeds, f_stride)):.2f} at the 8th harmonic")
tviz.show(tviz.plot_mode(modes, mode=1))

## 7. Cyclic loads: three missions, rainflow on the foot forces (Chronos), fatigue of the lower leg

Foot-force histories are synthesised per mission from the gait model: half-sine stance pulses at the stride
frequency with ±15 % random amplitude scatter, the longitudinal share, and the occasional drop (at least one per
simulated minute, so the short missions overstate their drops a little). Rainflow
counts them into blocks of the patterns `foot_z` and `foot_x`; the blocks scale to the mission duration.

In [ ]:
MISSIONS = {   # name: (gait, mass, minutes, drops per mission)
    "patrol walk": ("walk", M_DOG, 40.0, 2),
    "carry the pack to the site (trot)": ("trot", M_DOG + PAYLOAD_ALLOW_KG, 15.0, 1),
    "stairs with the pack": ("walk", M_DOG + PAYLOAD_ALLOW_KG, 8.0, 6),
}
SIM_S = 60.0
def foot_history(gait, mass, minutes, drops, seed, speed=None):
    beta = GAITS[gait]; v = {"walk": 0.6, "trot": 1.5, "bound": 2.6}[gait]
    fs = float(np.interp(v, speeds, f_stride)); dt = 1e-3
    tt = np.arange(0, SIM_S, dt); rng = np.random.default_rng(seed)
    Fz = np.zeros_like(tt); Fx = np.zeros_like(tt)
    peak = foot_peak(mass * G, beta); T = 1 / fs
    for k in range(int(SIM_S * fs)):
        on = (tt >= k * T) & (tt < k * T + beta * T)
        ph = (tt[on] - k * T) / (beta * T)
        amp = peak * rng.normal(1.0, 0.15)
        Fz[on] = amp * np.sin(np.pi * ph); Fx[on] = 0.3 * amp * np.cos(np.pi * ph) * (1.6 if "stairs" in gait else 1.0)
    # drops: a landing pulse replaces one stance per drop, scaled to the simulated minute
    for k in rng.choice(int(SIM_S * fs), size=max(1, round(drops / minutes * SIM_S / 60)), replace=False):   # drops in the simulated minute
        on = (tt >= k * T) & (tt < k * T + 0.05)
        Fz[on] = landing_force(mass)["peak per leg [N]"] * np.sin(np.pi * (tt[on] - k * T) / 0.05)
    return tt, Fz, Fx
hist = {name: foot_history(*spec, seed=i) for i, (name, spec) in enumerate(MISSIONS.items())}
fig, axes = plt.subplots(len(hist), 1, figsize=(13, 7), sharex=True)
for ax, (name, (tt, Fz, Fx)) in zip(axes, hist.items()):
    ax.plot(tt, Fz, lw=0.5, label="foot_z"); ax.plot(tt, Fx, lw=0.5, label="foot_x"); ax.set(title=name, ylabel="N", xlim=(0, 10)); ax.legend(fontsize=8); ax.grid(alpha=0.3)
axes[-1].set(xlabel="time [s] (first 10 s of the simulated minute)"); fig.tight_layout()

In [ ]:
spectra = {}
for name, (tt, Fz, Fx) in hist.items():
    minutes = MISSIONS[name][2]; scale = minutes * 60 / SIM_S
    blocks = []
    for pattern, series in (("foot_z", Fz), ("foot_x", Fx)):
        merged = {}
        for rng_, mean, count in chronos.rainflow(series[::4]):
            if rng_ < 2.0:
                continue
            key = (round(mean, 0), round(rng_ / 2, 0))
            merged[key] = merged.get(key, 0.0) + count * scale
        blocks += [chronos.Block(pattern, m_, a_, c_, f"{name}: {pattern} rainflow") for (m_, a_), c_ in merged.items()]
    spectra[name] = chronos.LoadSpectrum(name, minutes * 60, blocks, ["foot_z", "foot_x"], notes=f"gait model, {SIM_S:.0f} s simulated, scaled to {minutes:.0f} min")
    spectra[name].save(RUNS / f"spectrum_{name.split(' ')[0]}.json")
    fig = spectra[name].plot()
pd.DataFrame({k: {"blocks": len(sp.blocks), "cycles_per_mission": sp.total_cycles, "max_amplitude_N": max(b.amplitude for b in sp.blocks)} for k, sp in spectra.items()}).T.round(1)

In [ ]:
# unit cases on the lower leg: 100 N vertical and 100 N forward on the foot (ground frame, rotated into the leg frame)
unit_cases = {}
for pattern, (fx, fz) in {"foot_z": (0.0, 100.0), "foot_x": (100.0, 0.0)}.items():
    lx, lz = to_lower(fx, fz)
    um = lower_model([talos.Force("foot", fx=lx, fz=lz)], f"unit_{pattern}"); um.mesh(RUNS / f"unit_{pattern}")
    unit_cases[pattern] = (um.solve(RUNS / f"unit_{pattern}"), 100.0)
    print(pattern, unit_cases[pattern][0].status, round(unit_cases[pattern][0].metrics["max_von_mises"], 2), "MPa per 100 N")
CURVE = talos.FatigueCurve("CF-nylon (assumed)", sigma_f=200.0, b=-0.09, ultimate=160.0, source="assumed; coupon tests needed")
fatigue = {k: talos.assess_fatigue(unit_cases, sp.to_dict(), CURVE, workdir=RUNS / f"fatigue_{k.split(' ')[0]}") for k, sp in spectra.items()}
life = pd.DataFrame({k: {"mission_min": MISSIONS[k][2], "damage_per_mission": f.result.metrics["damage_per_pass"],
                         "missions_to_failure": f.result.metrics["passes_to_failure"], "hours_to_failure": f.result.metrics["hours_to_failure"]} for k, f in fatigue.items()}).T
life

In [ ]:
worst_m = life["damage_per_mission"].astype(float).idxmax()
tviz.show(tviz.plot_damage(fatigue[worst_m], unit_cases["foot_z"][0].artifacts["mesh"]))

In [ ]:
damage = {k: f.result.metrics["damage_per_pass"] for k, f in fatigue.items()}
hours = {k: MISSIONS[k][2] / 60 for k in MISSIONS}
usage = {"patrol walk": 0.5, "carry the pack to the site (trot)": 0.3, "stairs with the pack": 0.2}
rate = sum(usage[k] * damage[k] for k in usage) / sum(usage[k] * hours[k] for k in usage)
sim = chronos.simulate_life(damage, hours, usage, n_flights=5000, seed=0)
fig = sim.plot()
print(f"usage {usage}: damage per 1000 h {rate * 1000:.3g} -> " + (f"{1 / rate:.0f} h to failure" if rate > 1e-9 else "not life-limiting (> 1e9 h)"))
mixes = {"as above": usage, "stairs-only": {k: (1.0 if "stairs" in k else 0.0) for k in usage}, "patrol-only": {k: (1.0 if "patrol" in k else 0.0) for k in usage}}
pd.DataFrame({n: {"damage per 1000 h": sum(mx[k] * damage[k] for k in mx) / sum(mx[k] * hours[k] for k in mx) * 1000} for n, mx in mixes.items()}).T

## 8. The dog in motion: a trot, step by step, as a movie

The gait of section 2 as a sequence. Diagonal pairs move together (front-left with rear-right): during
its stance a foot stays on the ground and moves back relative to the hip at the body's speed; during the
swing it comes forward over a lifted arc. The inverse kinematics of section 3 turns every foot position
into hip and knee angles — plotted over one stride with the gait diagram — and the same angles place the
CAD legs frame by frame for the movie (`output/16_robot_dog/dog_trot.mp4`).

In [ ]:
import pyvista as pv
from vegeta.talos.viz import _write_video
OUT = Path("output/16_robot_dog"); OUT.mkdir(parents=True, exist_ok=True)
V_TROT = 1.5                                                     # m/s
F_STRIDE = float(np.interp(V_TROT, speeds, f_stride)); T_STRIDE = 1 / F_STRIDE
BETA = GAITS["trot"]
L_STANCE = V_TROT * 1000 * BETA * T_STRIDE                       # the foot's travel under the hip during one stance [mm]
SWING_LIFT = 60.0                                                # foot clearance in the swing [mm] (input)
PHASES = {"front-left": 0.0, "rear-right": 0.0, "front-right": 0.5, "rear-left": 0.5}
LEGS = {"front-left": (1, 1), "front-right": (1, -1), "rear-left": (-1, 1), "rear-right": (-1, -1)}
def foot_at(phase):
    """Foot position relative to the hip (x ahead, z down) and the stance flag at a stride phase in [0, 1)."""
    if phase < BETA:
        u = phase / BETA
        return L_STANCE / 2 * (1 - 2 * u), -H_STAND, True
    u = (phase - BETA) / (1 - BETA)
    return -L_STANCE / 2 + L_STANCE * u, -H_STAND + SWING_LIFT * math.sin(math.pi * u), False
def leg_angles(t_s, leg):
    phase = (t_s / T_STRIDE + PHASES[leg]) % 1.0
    x, z, stance = foot_at(phase)
    return leg_ik(x, z), stance, (x, z)
tt = np.linspace(0, T_STRIDE, 121)
fig, axes = plt.subplots(2, 1, figsize=(10, 6.5), sharex=True, gridspec_kw={"height_ratios": [3, 1]})
for leg, col in (("front-left", "#1565c0"), ("front-right", "#c62828")):
    ang = np.array([leg_angles(t_, leg)[0] for t_ in tt])
    axes[0].plot(tt, ang[:, 0], color=col, label=f"{leg}: hip"); axes[0].plot(tt, ang[:, 1], "--", color=col, label=f"{leg}: knee")
axes[0].set(ylabel="joint angle [deg]", title=f"one trot stride at {V_TROT} m/s: {F_STRIDE:.2f} Hz, stance {L_STANCE:.0f} mm, duty {BETA}"); axes[0].legend(fontsize=8, ncol=2); axes[0].grid(alpha=0.3)
for j, leg in enumerate(LEGS):
    st = np.array([leg_angles(t_, leg)[1] for t_ in tt])
    axes[1].fill_between(tt, j - 0.4, j + 0.4, where=st, color="#3f4347", step="mid")
axes[1].set(yticks=range(4), yticklabels=list(LEGS), xlabel="time [s]", title="gait diagram (dark = foot on the ground)"); axes[1].grid(alpha=0.3, axis="x")
fig.tight_layout()

In [ ]:
body_m, upper_m, lower_m = dviz.to_pyvista(body), dviz.to_pyvista(upper), dviz.to_pyvista(lower)
ground = pv.Plane(center=(2200, 0, 0), direction=(0, 0, 1), i_size=6000, j_size=1200)
ticks = pv.MultiBlock([pv.Line((x_, -600, 1), (x_, 600, 1)) for x_ in np.arange(-1000, 5200, 250)]).combine()
def dog_frame(t_s, size=(1100, 650)):
    x_body = V_TROT * 1000 * t_s
    pl = pv.Plotter(off_screen=True, window_size=size)
    pl.add_mesh(ground, color="#5e6367"); pl.add_mesh(ticks, color="#8b9096", line_width=2)
    pl.add_mesh(body_m.copy().translate((x_body, 0, 0), inplace=True), color="#cfd6dd", smooth_shading=True)
    for leg, (sx, sy) in LEGS.items():
        (a1_, a2_), stance, _ = leg_angles(t_s, leg)
        hip = np.array([x_body + sx * p["hip_x"], sy * y_leg, H_STAND])
        knee = hip + np.array([-L1 * math.sin(math.radians(a1_)), 0.0, -L1 * math.cos(math.radians(a1_))])
        pl.add_mesh(upper_m.copy().rotate_y(a1_, inplace=True).translate(hip, inplace=True), color="#3f4347", smooth_shading=True)
        pl.add_mesh(lower_m.copy().rotate_y(-a2_, inplace=True).translate(knee, inplace=True), color="#3f4347" if stance else "#9a9f8a", smooth_shading=True)
    pl.add_text(f"trot {V_TROT} m/s   t = {t_s:4.2f} s   stride {t_s / T_STRIDE:4.1f}", font_size=10)
    pl.camera_position = [(x_body + 1100, -1700, 650), (x_body, 0, 180), (0, 0, 1)]
    img = pl.screenshot(return_img=True); pl.close()
    return img
FPS, SECONDS = 20, 3 * T_STRIDE
frames = [dog_frame(t_) for t_ in tqdm(np.arange(0, SECONDS, 1 / FPS))]
movie = _write_video(frames, OUT / "dog_trot.mp4", FPS)
fig, axes = plt.subplots(2, 3, figsize=(15, 6))
for ax, j in zip(axes.flat, np.linspace(0, int(T_STRIDE * FPS) - 1, 6).astype(int)):
    ax.imshow(frames[j]); ax.set_axis_off(); ax.set_title(f"t = {j / FPS:.2f} s, phase {(j / FPS) / T_STRIDE:.2f}")
fig.suptitle("one stride of the trot"); fig.tight_layout()
print(f"movie: {movie} ({len(frames)} frames, {SECONDS:.1f} s = 3 strides)")

## 9. Steps on open terrain: a walk over rough ground, joint by joint, as a movie

Open terrain: long ground waves with rocks on them (a seeded field, `terrain(x, y)`). The dog walks it at
0.6 m/s with a lateral-sequence walk (rear-left, front-left, rear-right, front-right; duty 0.75, three feet
down at any time). Every swing aims at a landing point ahead of the hip **on the ground**; a stance foot
stays where it landed while the body passes over it; the body rides at the standing height over the
ground under its hips, **pitches** with the slope between front and rear and **rolls** with the slope
between left and right. Every leg has its three joints: the **hip roll** swings the leg plane out to where
the foot is, the **hip pitch** and the **knee** come from the inverse kinematics in that plane. If a foot
cannot reach, it is pulled in along the hip line and flagged.

Step by step: each landing is a row — where the foot went down, the ground there, the three joint angles
and the three joint torques at that instant (three feet carry the weight; the ground force is vertical) —
followed by the twelve joint-angle traces, the gait diagram, and the movie
(`output/16_robot_dog/dog_walk_terrain.mp4`).

In [ ]:
rng_t = np.random.default_rng(3)
ROCKS = [(rng_t.uniform(400, 3800), rng_t.uniform(-320, 320), rng_t.uniform(30, 70), rng_t.uniform(80, 140)) for _ in range(14)]
def terrain(x, y):
    """Ground height [mm] at (x, y) [mm]: two long waves and gaussian rocks."""
    z = 45 * np.sin(2 * np.pi * x / 2600) + 20 * np.sin(2 * np.pi * (x + 0.6 * y) / 900)
    for rx, ry, rh, rw in ROCKS:
        z = z + rh * np.exp(-((x - rx) ** 2 + (y - ry) ** 2) / rw**2)
    return z
def Rx(a): c_, s_ = math.cos(a), math.sin(a); return np.array([[1, 0, 0], [0, c_, -s_], [0, s_, c_]])
def Ry(a): c_, s_ = math.cos(a), math.sin(a); return np.array([[c_, 0, s_], [0, 1, 0], [-s_, 0, c_]])
V_WALK, BETA_W = 0.6, GAITS["walk"]
F_W = float(np.interp(V_WALK, speeds, f_stride)); T_W = 1 / F_W
STRIDE_W = V_WALK * 1000 * T_W
PHASES_W = {"rear-left": 0.0, "front-left": 0.25, "rear-right": 0.5, "front-right": 0.75}
DT = 1 / 30
T_END = 4.5 * T_W
times = np.arange(0, T_END, DT)
def body_pose(t_s):
    """Body centre, its rotation (pitch nose-up with the front/rear slope, roll with the left/right slope) and the hips in world coordinates."""
    xb = V_WALK * 1000 * t_s
    zf, zr = float(terrain(xb + p["hip_x"], 0.0)), float(terrain(xb - p["hip_x"], 0.0))
    zl, zrt = float(terrain(xb, y_leg)), float(terrain(xb, -y_leg))
    th = math.atan2(zf - zr, 2 * p["hip_x"]); ph = math.atan2(zl - zrt, 2 * y_leg)
    R = Ry(-th) @ Rx(ph)                                                      # body axes in the world (pyvista: rotate_x(ph) then rotate_y(-th))
    centre = np.array([xb, 0.0, (zf + zr) / 2 + H_STAND])
    hips = {leg: centre + R @ np.array([sx * p["hip_x"], sy * y_leg, 0.0]) for leg, (sx, sy) in LEGS.items()}
    return centre, R, th, ph, hips
def leg_joints(foot_world, hip_world, R):
    """Hip roll, hip pitch, knee [deg] for a foot in the world; also the foot in the body frame and the reach flag."""
    rel = R.T @ (foot_world - hip_world)                                     # into the body frame
    roll = math.atan2(rel[1], -rel[2]) if rel[2] < 0 else math.atan2(rel[1], 1e-6)
    zp = -math.hypot(rel[1], rel[2]); xp = rel[0]                             # in the rolled leg plane
    ik = leg_ik(xp, zp); ok = ik is not None
    if not ok:
        r = math.hypot(xp, zp); k = (L1 + L2 - 2) / r; ik = leg_ik(xp * k, zp * k)
    return (math.degrees(roll), ik[0], ik[1]), rel, ok
def leg_torques_3(roll_deg, a1_, a2_, rel, F_body):
    """Torques [N m] at the hip roll, hip pitch and knee for a ground force F (body frame) on a foot at rel (body frame)."""
    tau_roll = float(np.cross(rel, F_body)[0]) / 1000
    Fp = Rx(-math.radians(roll_deg)) @ F_body                                 # force in the leg plane
    xp, zp = rel[0], -math.hypot(rel[1], rel[2])
    knee_p = np.array([-L1 * math.sin(math.radians(a1_)), -L1 * math.cos(math.radians(a1_))])
    tau_hip = (xp * Fp[2] - zp * Fp[0]) / 1000
    tau_knee = ((xp - knee_p[0]) * Fp[2] - (zp - knee_p[1]) * Fp[0]) / 1000
    return tau_roll, tau_hip, tau_knee
# step the feet through time: a stance foot is fixed in the world, a swing foot flies to its landing point on the ground
JOINT_NAMES = ["hip roll", "hip pitch", "knee"]
feet, stance_flags, joints_w, steps, unreachable = {}, {}, {}, [], 0
state = {}
_, R0, _, _, hips0 = body_pose(0.0)
for leg, (sx, sy) in LEGS.items():
    x0 = hips0[leg][0] + STRIDE_W * BETA_W / 2 * (1 - 2 * PHASES_W[leg] / BETA_W) if PHASES_W[leg] < BETA_W else hips0[leg][0]
    state[leg] = {"foot": np.array([x0, hips0[leg][1], float(terrain(x0, hips0[leg][1]))]), "lift": None, "land": None, "was_stance": True}
    feet[leg], stance_flags[leg], joints_w[leg] = [], [], []
for t_s in times:
    centre, R, th, ph, hips = body_pose(t_s)
    for leg, (sx, sy) in LEGS.items():
        phase = (t_s / T_W + PHASES_W[leg]) % 1.0; st = state[leg]
        if phase < BETA_W:                                                    # stance: the foot stays put
            if st["lift"] is not None:                                        # just landed
                st["foot"] = st["land"].copy(); st["lift"] = None
            foot, in_stance = st["foot"], True
        else:
            u = (phase - BETA_W) / (1 - BETA_W)
            if st["lift"] is None:                                            # swing starts: aim at the landing point, half a stance ahead of the hip, on the ground
                st["lift"] = st["foot"].copy()
                _, _, _, _, hips_land = body_pose(t_s + (1 - u) * (1 - BETA_W) * T_W)
                xl, yl = hips_land[leg][0] + STRIDE_W * BETA_W / 2, hips_land[leg][1]
                st["land"] = np.array([xl, yl, float(terrain(xl, yl))])
            lift, land = st["lift"], st["land"]
            foot = lift + (land - lift) * u
            clear = max(0.0, float(terrain(foot[0], foot[1])) - foot[2] + 15.0)   # clear a rock on the way
            foot = foot + np.array([0, 0, (SWING_LIFT + clear) * math.sin(math.pi * u)])
            in_stance = False
        (roll, a1_, a2_), rel, ok = leg_joints(foot, hips[leg], R)
        unreachable += 0 if ok else 1
        if in_stance and not st["was_stance"]:                                # a landing: one step, with the joint loads at that instant
            F_body = R.T @ np.array([0.0, 0.0, W_DOG / 3])
            tr, thp, tk = leg_torques_3(roll, a1_, a2_, rel, F_body)
            steps.append({"t [s]": t_s, "leg": leg, "x [mm]": foot[0], "y [mm]": foot[1], "ground z [mm]": foot[2], "body pitch [deg]": math.degrees(th), "body roll [deg]": math.degrees(ph),
                          "hip roll [deg]": roll, "hip pitch [deg]": a1_, "knee [deg]": a2_, "τ roll [Nm]": tr, "τ hip [Nm]": thp, "τ knee [Nm]": tk,
                          "worst joint / peak motor": max(abs(tr), abs(thp), abs(tk)) / MOTOR["peak_Nm"], "reachable": ok})
        st["was_stance"] = in_stance
        feet[leg].append(foot.copy()); stance_flags[leg].append(in_stance); joints_w[leg].append((roll, a1_, a2_))
feet = {k: np.array(v) for k, v in feet.items()}; stance_flags = {k: np.array(v) for k, v in stance_flags.items()}; joints_w = {k: np.array(v) for k, v in joints_w.items()}
steps_tab = pd.DataFrame(steps); steps_tab.index = [f"step {i + 1}" for i in range(len(steps_tab))]
print(f"walk {V_WALK} m/s: {F_W:.2f} Hz, stride {STRIDE_W:.0f} mm, {len(steps_tab)} steps in {T_END:.1f} s, {unreachable} unreachable samples")
steps_tab.round(2)

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(14, 11), gridspec_kw={"height_ratios": [2.2, 2, 1.2]})
xs_ = np.linspace(-300, V_WALK * 1000 * T_END + 600, 400)
ax = axes[0, 0]
ax.fill_between(xs_, -120, terrain(xs_, y_leg), color="#8d7b68", alpha=0.6, label="ground under the left feet")
for leg, col in (("front-left", "#1565c0"), ("rear-left", "#2e7d32")):
    f = feet[leg]; ax.plot(f[:, 0], f[:, 2], color=col, lw=1.2, label=f"{leg} foot path"); ax.plot(f[stance_flags[leg], 0], f[stance_flags[leg], 2], ".", color=col, ms=3)
bodies = np.array([body_pose(t_)[0] for t_ in times]); ax.plot(bodies[:, 0], bodies[:, 2], "k--", lw=0.8, label="body centre")
for i, r_ in steps_tab.iterrows():
    if "left" in r_["leg"]:
        ax.annotate(i.split()[1], (r_["x [mm]"], r_["ground z [mm]"] - 25), fontsize=7, ha="center")
ax.set(ylabel="z [mm]", title="foot paths over the ground (step numbers at the landings)"); ax.legend(fontsize=7); ax.set_aspect("equal"); ax.grid(alpha=0.3)
ax = axes[0, 1]
ax.plot(times, [math.degrees(body_pose(t_)[2]) for t_ in times], label="body pitch"); ax.plot(times, [math.degrees(body_pose(t_)[3]) for t_ in times], label="body roll")
ax.set(ylabel="deg", title="body attitude from the ground under the hips"); ax.legend(fontsize=8); ax.grid(alpha=0.3)
for ax, (leg, col) in zip(axes[1], (("front-left", "#1565c0"), ("rear-left", "#2e7d32"))):
    for j, (jn, ls) in enumerate(zip(JOINT_NAMES, ("-.", "-", "--"))):
        ax.plot(times, joints_w[leg][:, j], ls, color=col, label=f"{leg}: {jn}")
    for leg2, col2 in (("front-right", "#c62828"), ("rear-right", "#6d4c41")):
        if leg2.split("-")[0] == leg.split("-")[0]:
            for j, (jn, ls) in enumerate(zip(JOINT_NAMES, ("-.", "-", "--"))):
                ax.plot(times, joints_w[leg2][:, j], ls, color=col2, label=f"{leg2}: {jn}", lw=0.9)
    ax.set(ylabel="joint angle [deg]", title=f"{leg.split('-')[0]} legs: all three joints"); ax.legend(fontsize=7, ncol=2); ax.grid(alpha=0.3)
ax = axes[2, 0]
for j, leg in enumerate(LEGS):
    ax.fill_between(times, j - 0.4, j + 0.4, where=stance_flags[leg], color="#3f4347", step="mid")
ax.set(yticks=range(4), yticklabels=list(LEGS), xlabel="time [s]", title="gait diagram (dark = foot on the ground)"); ax.grid(alpha=0.3, axis="x")
ax = axes[2, 1]
steps_tab[["τ roll [Nm]", "τ hip [Nm]", "τ knee [Nm]"]].abs().plot.bar(ax=ax, width=0.8); ax.axhline(MOTOR["peak_Nm"], color="k", ls="-.", lw=0.8); ax.axhline(MOTOR["continuous_Nm"], color="k", ls=":", lw=0.8)
ax.set(ylabel="N m", title="joint torques at each landing (three feet down)"); ax.tick_params(axis="x", labelsize=7); ax.legend(fontsize=7); ax.grid(alpha=0.3, axis="y")
fig.tight_layout()

In [ ]:
gx, gy = np.meshgrid(np.linspace(-600, V_WALK * 1000 * T_END + 900, 160), np.linspace(-700, 700, 60))
ground_t = pv.StructuredGrid(gx, gy, terrain(gx, gy)); ground_t["z"] = terrain(gx, gy).ravel(order="F")
def terrain_frame(idx, size=(1100, 650)):
    t_s = times[idx]; centre, R, th, ph, hips = body_pose(t_s); th_deg, ph_deg = math.degrees(th), math.degrees(ph)
    pl = pv.Plotter(off_screen=True, window_size=size)
    pl.add_mesh(ground_t, scalars="z", cmap="copper", show_scalar_bar=False, smooth_shading=True)
    pl.add_mesh(body_m.copy().translate((0, 0, -H_STAND), inplace=True).rotate_x(ph_deg, inplace=True).rotate_y(-th_deg, inplace=True).translate(centre, inplace=True), color="#cfd6dd", smooth_shading=True)
    for leg, (sx, sy) in LEGS.items():
        roll, a1_, a2_ = joints_w[leg][idx]; hip = hips[leg]
        R_leg = R @ Rx(math.radians(roll))                                    # the leg plane in the world
        knee = hip + R_leg @ np.array([-L1 * math.sin(math.radians(a1_)), 0.0, -L1 * math.cos(math.radians(a1_))])
        pl.add_mesh(upper_m.copy().rotate_y(a1_, inplace=True).rotate_x(roll + ph_deg, inplace=True).rotate_y(-th_deg, inplace=True).translate(hip, inplace=True), color="#3f4347", smooth_shading=True)
        pl.add_mesh(lower_m.copy().rotate_y(-a2_, inplace=True).rotate_x(roll + ph_deg, inplace=True).rotate_y(-th_deg, inplace=True).translate(knee, inplace=True), color="#3f4347" if stance_flags[leg][idx] else "#9a9f8a", smooth_shading=True)
    pl.add_text(f"walk {V_WALK} m/s on open terrain   t = {t_s:4.2f} s   pitch {th_deg:+4.1f} deg  roll {ph_deg:+4.1f} deg", font_size=10)
    pl.camera_position = [(centre[0] + 1300, -1900, centre[2] + 500), (centre[0], 0, centre[2] - 150), (0, 0, 1)]
    img = pl.screenshot(return_img=True); pl.close()
    return img
frames_t = [terrain_frame(i) for i in tqdm(range(len(times)))]
movie_t = _write_video(frames_t, OUT / "dog_walk_terrain.mp4", int(round(1 / DT)))
fig, axes = plt.subplots(2, 3, figsize=(15, 6))
for ax, j in zip(axes.flat, np.linspace(0, len(frames_t) - 1, 6).astype(int)):
    ax.imshow(frames_t[j]); ax.set_axis_off(); ax.set_title(f"t = {times[j]:.2f} s")
fig.suptitle("the walk over open terrain"); fig.tight_layout()
print(f"movie: {movie_t} ({len(frames_t)} frames, {T_END:.1f} s)")

## 10. Print the upper leg (Mellonia)

Flat on the bed (the plate is built in the XZ plane, so it is rotated 90° about x): the layers run along the
leg, in the direction of the bending stress, which is what the flat-orientation material values assume.

In [ ]:
prn = mellonia.slice_stl(cad["upper_leg"].artifacts["stl"], GENERIC_PLA_0_2MM, mellonia.Orientation(rotate_x=90), RUNS / "print_upper_leg")
print(prn)
if prn.ok:
    mviz.show(mviz.plot_toolpath(prn))

## 11. Export

In [ ]:
summary_doc = {"source": "16_robot_dog", "design": {"file": "designs/robot_dog.py", "parameters": p},
           "mass_kg": M_DOG, "payload_allowance_kg": PAYLOAD_ALLOW_KG, "standing_height_mm": H_STAND, "deck_z_mm": DECK_Z,
           "deck": {"length_mm": p["deck_length"], "width_mm": p["deck_width"], "hole_pitch_mm": p["deck_hole_pitch"], "hole_diameter_mm": p["deck_hole_diameter"]},
           "stance_width_mm": 2 * y_leg + p["upper_leg_thickness"], "wheelbase_mm": WHEELBASE, "cg_height_loaded_mm": H_CG_LOADED,
           "stairs": {"riser_mm": RISER, "tread_mm": TREAD, "tipping_margin_loaded_mm": stairs.loc["with payload, dry concrete", "tipping margin [mm]"]},
           "gait_peak_foot_N": {g: foot_peak(W_LOADED, b) for g, b in GAITS.items()}, "landing_peak_N": landing.loc["with payload", "peak per leg [N]"],
           "lower_leg_modes_hz": freqs, "motor": MOTOR, "joint_torques": torques.to_dict(orient="index"),
           "static": static.to_dict(orient="index"), "life": life.to_dict(orient="index"), "damage_per_1000h": rate * 1000,
           "trot": {"speed_m_s": V_TROT, "stride_hz": F_STRIDE, "stance_mm": L_STANCE, "movie": str(movie)},
           "terrain_walk": {"speed_m_s": V_WALK, "stride_hz": F_W, "stride_mm": STRIDE_W, "unreachable_samples": unreachable, "steps": steps_tab.to_dict(orient="index"), "movie": str(movie_t)}}
(RUNS / "robot_dog.json").write_text(json.dumps(summary_doc, indent=2, default=float))
print("written:", sorted(x.name for x in RUNS.iterdir())[:14], "...")

**What the numbers say:** every gait and the stairs leave both leg links above SF 3; the one case below
SF 1.5 is the loaded drop landing, which takes the upper leg plate past yield (SF ≈ 0.8) — so either the
plate grows at the knee boss (width and thickness, a parameter study on `upper_leg_width` /
`upper_leg_thickness`), or the controller limits drops with the payload on. Fatigue is not life-limiting.

**Next steps an engineer would take:** measure the real actuators' torque–speed curves (the stance sweep
above assumes a constant peak), add the hip-roll joint and a lateral case (a sideways push on the stairs),
replace the synthetic foot histories with force-plate recordings of a real trot, and run the upper-leg
revision for the drop case in a `vegeta.core` workspace, or as a campaign with the drop-landing safety
factor as the objective.